# Notebook 07: Feature Expansion

The baseline model used five variables: maternal age, education level, wealth index, 
residence, and household size. Those variables explain some of the socioeconomic 
gradient in stunting but leave the child level completely uncharacterised. A model 
with no information about child age, sex, or birth history is fundamentally limited 
regardless of how sophisticated the algorithm is.

This notebook expands the modeling dataset to 24 columns by pulling 12 additional 
predictors from the cleaned KR file. Every variable selected corresponds to a named 
causal pathway in the UNICEF Conceptual Framework for Child Undernutrition. Variables 
are prefixed to make their theoretical position explicit throughout all downstream work:

    imm_  immediate causes operating at the child level
    und_  underlying causes at the maternal and household level
    str_  structural causes at the community and societal level
    id_   identifiers not used as predictors
    wt_   sampling weight required by the DHS survey design

The sampling weight sv005 is carried forward on every record. The 2024 MDHS used a 
stratified two-stage cluster sampling design and all prevalence estimates and model 
outputs must be interpretable against that design.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_ROOT  = Path("/Users/frack/Documents/PhD Data Science/malawi-dhs-2024-geoai")
DATA_INTERIM  = PROJECT_ROOT / "data" / "interim"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

df_kr    = pd.read_parquet(DATA_INTERIM / "kr_clean.parquet")
df_model = pd.read_parquet(DATA_PROCESSED / "model_dataset.parquet")

print("KR file:", df_kr.shape)
print("Current model dataset:", df_model.shape)

KR file: (5415, 1210)
Current model dataset: (5415, 9)


## Extracting new variables

Twelve variables are pulled from kr_clean covering three gaps in the current feature set.

Child characteristics: age in months is the single most important missing variable 
given how sharply stunting accumulates between 6 and 24 months. Sex is included because 
male children show consistently higher stunting rates in Malawi and across sub-Saharan 
Africa. Birth order and birth interval capture the dilution effect and maternal 
nutritional depletion between pregnancies.

Maternal biology: weight and height are direct measures of intergenerational 
transmission. A stunted mother is more likely to have a stunted child through placental 
insufficiency and reduced birth weight. Education in years adds granularity beyond the 
education level category already in the dataset.

Illness and birth context: recent diarrhea captures the gut-growth pathway where 
enteric infection disrupts nutrient absorption directly. Size at birth proxies 
intrauterine growth restriction. Region and religion capture agro-ecological variation 
and community-level dietary and health-seeking norms respectively.

DHS coding note: maternal weight v437 and height v438 are stored as integers in tenths 
of a unit, so 516 means 51.6 kg. Both are divided by 10 on extraction.

In [2]:
new_vars = [
    "v001", "v002", "bidx", "sv005",
    "hw1",    # child age in months
    "b4",     # child sex
    "bord",   # birth order
    "b11",    # preceding birth interval in months
    "m18",    # size at birth
    "h11",    # diarrhea in last two weeks
    "v133",   # maternal education in years
    "v437",   # maternal weight, divide by 10 for kg
    "v438",   # maternal height, divide by 10 for cm
    "v201",   # total children ever born
    "v024",   # region
    "v130"    # religion
]

df_new = df_kr[new_vars].copy()

df_new["v437"] = pd.to_numeric(df_new["v437"], errors="coerce") / 10
df_new["v438"] = pd.to_numeric(df_new["v438"], errors="coerce") / 10
df_new["v133"] = pd.to_numeric(df_new["v133"], errors="coerce")

print("Extracted shape:", df_new.shape)
print("\nMissingness:")
print(df_new.isnull().mean().round(3))

Extracted shape: (5415, 16)

Missingness:
v001     0.000
v002     0.000
bidx     0.000
sv005    0.000
hw1      0.000
b4       0.000
bord     0.000
b11      0.306
m18      0.000
h11      0.000
v133     0.000
v437     0.008
v438     0.007
v201     0.000
v024     0.000
v130     0.000
dtype: float64


## Handling missingness

Four variables need documented decisions.

Birth interval b11 is missing for 30.6% of children. These are all first-borns for 
whom no preceding birth exists, so this is structural absence not data loss. We create 
a binary flag to preserve the information that being first-born is itself a risk factor, 
then fill the missing values with the median interval among non-first-borns.

Size at birth m18 has 37.3% NaN values meaning the mother had no health card and could 
not recall. We treat this as a category called not_reported rather than imputing. The 
absence of a health card is itself a signal about healthcare access that should not be 
silently erased.

Maternal weight and height each have under 1% missing from genuine field measurement 
gaps. Median imputation is appropriate here.

Diarrhea h11 has four records coded as don't know. These collapse into no.

In [3]:
# birth interval: flag first-borns then fill missing with median
df_new["imm_first_born"] = df_new["b11"].isnull().astype(int)
b11_median = df_new["b11"].median()
df_new["b11"] = df_new["b11"].fillna(b11_median)

# size at birth: treat missing as a meaningful category
df_new["m18"] = (df_new["m18"]
                 .astype(str)
                 .str.strip()
                 .str.lower()
                 .replace("nan", "not_reported"))

# diarrhea: four don't know records collapse into no
df_new["h11"] = (df_new["h11"]
                 .astype(str)
                 .str.strip()
                 .str.lower()
                 .replace("don't know", "no"))

# maternal anthropometry: median imputation for small field gaps
df_new["v437"] = df_new["v437"].fillna(df_new["v437"].median())
df_new["v438"] = df_new["v438"].fillna(df_new["v438"].median())

print("Missingness after handling:")
print(df_new.isnull().mean().round(3))

print("\nFirst-born flag:")
print(df_new["imm_first_born"].value_counts())

print("\nSize at birth:")
print(df_new["m18"].value_counts())

print("\nDiarrhea:")
print(df_new["h11"].value_counts())

Missingness after handling:
v001              0.0
v002              0.0
bidx              0.0
sv005             0.0
hw1               0.0
b4                0.0
bord              0.0
b11               0.0
m18               0.0
h11               0.0
v133              0.0
v437              0.0
v438              0.0
v201              0.0
v024              0.0
v130              0.0
imm_first_born    0.0
dtype: float64

First-born flag:
imm_first_born
0    3757
1    1658
Name: count, dtype: int64

Size at birth:
m18
not_reported            2020
average                 1610
larger than average     1031
smaller than average     375
very large               228
very small                94
don't know                57
Name: count, dtype: int64

Diarrhea:
h11
no                     4188
yes, last two weeks    1227
Name: count, dtype: int64


## Merging new features into the model dataset

Merging on cluster and household alone inflates the dataset because households can 
have multiple children. Adding bidx resolves most cases but a small number of twin 
pairs share the same cluster, household, and bidx. Adding the HAZ score as a fourth 
key breaks these ties since each child has a distinct measured value.

The model dataset was saved in notebook 03 without bidx, so we recover it positionally 
from kr_clean. This is safe because df_model was built from kr_clean without any row 
reordering, which we verify by checking that hw70 values match before proceeding.

Three records are dropped after the merge. These are twin pairs where even the HAZ 
score is identical, making the records indistinguishable by any available identifier. 
Three records out of 5415 is negligible and is documented here.

In [4]:
# recover bidx positionally and verify row order is consistent
df_model_keyed = df_model.copy()
df_model_keyed["bidx"] = df_kr["bidx"].values

mismatch = (df_model_keyed["hw70"] != df_kr["hw70"].values).sum()
print(f"hw70 positional mismatch: {mismatch} (must be 0 to proceed)")
assert mismatch == 0, "Row order mismatch between df_model and df_kr"

# bring hw70 into df_new as a tiebreaker for twin records
df_new_keyed = df_new.copy()
df_new_keyed["hw70"] = df_kr["hw70"].values

# merge on the four-part key
df_expanded = df_model_keyed.merge(
    df_new_keyed,
    on=["v001", "v002", "bidx", "hw70"],
    how="left"
)

# drop twin records that are indistinguishable by any available key
before = len(df_expanded)
df_expanded = df_expanded.drop_duplicates(subset=["v001", "v002", "bidx", "hw70"])
after = len(df_expanded)

print(f"Rows dropped in dedup: {before - after}")
print("Shape after merge:", df_expanded.shape)

hw70 positional mismatch: 0 (must be 0 to proceed)
Rows dropped in dedup: 3
Shape after merge: (5414, 24)


## Renaming, validating, and saving

All columns are renamed to readable English with their framework prefix so the 
variable list is self-documenting at every downstream stage. Three checks run before 
saving: row count within expected range, no unexpected nulls, sampling weight present 
on every record.

The expanded dataset saves as model_dataset_v2.parquet. The original 
model_dataset.parquet stays untouched as the baseline reference.

In [5]:
rename_map = {
    "v001"   : "id_cluster",
    "v002"   : "id_household",
    "bidx"   : "id_child_index",
    "hw70"   : "haz_score",
    "stunted": "outcome_stunted",
    "sv005"  : "wt_sample_weight",
    "v012_x" : "und_maternal_age",
    "v106_x" : "und_maternal_edu_level",
    "v190_x" : "str_wealth_index",
    "v025_x" : "str_residence",
    "hv009"  : "und_household_size",
    "hw1"    : "imm_child_age_months",
    "b4"     : "imm_child_sex",
    "bord"   : "imm_birth_order",
    "b11"    : "imm_birth_interval",
    "m18"    : "imm_size_at_birth",
    "h11"    : "imm_had_diarrhea",
    "v133"   : "und_maternal_edu_years",
    "v437"   : "und_maternal_weight_kg",
    "v438"   : "und_maternal_height_cm",
    "v201"   : "und_total_children",
    "v024"   : "str_region",
    "v130"   : "str_religion"
}

df_expanded = df_expanded.rename(columns=rename_map)
cols_final  = list(rename_map.values()) + ["imm_first_born"]
df_expanded = df_expanded[cols_final]

# row count: 3 twin records removed, anything between 5412 and 5415 is acceptable
assert len(df_expanded) in [5412, 5413, 5414, 5415], \
    f"Unexpected row count: {len(df_expanded)}"

print("=== Row count ===")
print(len(df_expanded), "(3 duplicate records removed, documented in block 4)")

print("\n=== Missingness ===")
print(df_expanded.isnull().mean().round(3))

print("\n=== Sampling weight ===")
print("Missing weights:", df_expanded["wt_sample_weight"].isnull().sum())

print("\n=== Outcome distribution ===")
vc  = df_expanded["outcome_stunted"].value_counts()
vcp = df_expanded["outcome_stunted"].value_counts(normalize=True).round(3)
print(pd.concat([vc, vcp], axis=1, keys=["count", "proportion"]))

print("\n=== Numeric summaries ===")
num_cols = [
    "imm_child_age_months", "imm_birth_interval", "imm_birth_order",
    "und_maternal_age", "und_maternal_edu_years",
    "und_maternal_weight_kg", "und_maternal_height_cm",
    "und_total_children", "und_household_size"
]
print(df_expanded[num_cols].describe().round(2))

print("\n=== Categorical value counts ===")
cat_cols = [
    "imm_child_sex", "imm_size_at_birth", "imm_had_diarrhea",
    "und_maternal_edu_level", "str_wealth_index",
    "str_residence", "str_region", "str_religion"
]
for col in cat_cols:
    print(f"\n{col}:")
    print(df_expanded[col].value_counts())

# convert any remaining category dtypes before saving
for col in df_expanded.select_dtypes(["category"]).columns:
    df_expanded[col] = df_expanded[col].astype(str)

df_expanded.to_parquet(DATA_PROCESSED / "model_dataset_v2.parquet", index=False)
print("\nSaved: model_dataset_v2.parquet")
print("Final shape:", df_expanded.shape)
print("\nFinal columns:")
for col in df_expanded.columns:
    print(" ", col)

=== Row count ===
5414 (3 duplicate records removed, documented in block 4)

=== Missingness ===
id_cluster                0.0
id_household              0.0
id_child_index            0.0
haz_score                 0.0
outcome_stunted           0.0
wt_sample_weight          0.0
und_maternal_age          0.0
und_maternal_edu_level    0.0
str_wealth_index          0.0
str_residence             0.0
und_household_size        0.0
imm_child_age_months      0.0
imm_child_sex             0.0
imm_birth_order           0.0
imm_birth_interval        0.0
imm_size_at_birth         0.0
imm_had_diarrhea          0.0
und_maternal_edu_years    0.0
und_maternal_weight_kg    0.0
und_maternal_height_cm    0.0
und_total_children        0.0
str_region                0.0
str_religion              0.0
imm_first_born            0.0
dtype: float64

=== Sampling weight ===
Missing weights: 0

=== Outcome distribution ===
                 count  proportion
outcome_stunted                   
0                 3482  

Two things to note as working in  notebook 08 from this output.

1. str_wealth_index has 292 records showing as "nan" string. These are not actual nulls which is why they passed the missingness check. They are string-encoded missing values from the Stata merge in notebook 03. We fix this in notebook 08 by replacing the string "nan" with a proper NaN then deciding how to handle it.


2. imm_size_at_birth still has 57 records as "don't know" rather than collapsing into not_reported. The apostrophe in the Stata-encoded string is a different character than what our replace used. We fix this in notebook 08 with a contains-based approach instead of exact match.

In [7]:
assert len(df_expanded) == 5415, f"Row count changed to {len(df_expanded)}"

print("=== Missingness ===")
print(df_expanded.isnull().mean().round(3))

print("\n=== Sampling weight check ===")
print("Missing weights:", df_expanded["wt_sample_weight"].isnull().sum())

print("\n=== Outcome distribution ===")
vc = df_expanded["outcome_stunted"].value_counts()
vcp = df_expanded["outcome_stunted"].value_counts(normalize=True).round(3)
print(pd.concat([vc, vcp], axis=1, keys=["count", "proportion"]))

print("\n=== Numeric summaries ===")
num_cols = [
    "imm_child_age_months", "imm_birth_interval", "imm_birth_order",
    "und_maternal_age", "und_maternal_edu_years",
    "und_maternal_weight_kg", "und_maternal_height_cm",
    "und_total_children", "und_household_size"
]
print(df_expanded[num_cols].describe().round(2))

# convert any remaining category dtypes before saving
for col in df_expanded.select_dtypes(["category"]).columns:
    df_expanded[col] = df_expanded[col].astype(str)

df_expanded.to_parquet(DATA_PROCESSED / "model_dataset_v2.parquet", index=False)
print("\nSaved: model_dataset_v2.parquet")
print("Final shape:", df_expanded.shape)

AssertionError: Row count changed to 5723

In [8]:
# check if v001+v002+bidx is unique in both files
print("df_kr duplicates on v001+v002+bidx:", 
      df_kr.duplicated(subset=["v001","v002","bidx"]).sum())

print("df_model duplicates on v001+v002:", 
      df_model.duplicated(subset=["v001","v002"]).sum())

# check how many rows df_model and df_kr share the same v001+v002 combination
kr_keys = df_kr[["v001","v002","bidx"]].drop_duplicates()
print("\nUnique v001+v002+bidx in kr_clean:", len(kr_keys))
print("Rows in df_model:", len(df_model))

# look at a household with multiple children
multi = df_kr[df_kr.duplicated(subset=["v001","v002"], keep=False)]
print("\nSample multi-child household in kr:")
print(multi[["v001","v002","bidx"]].head(10))

# check if df_model has a unique row identifier at all
print("\ndf_model columns:", df_model.columns.tolist())
print("df_model head:")
print(df_model[["v001","v002"]].head(10))

df_kr duplicates on v001+v002+bidx: 148
df_model duplicates on v001+v002: 823

Unique v001+v002+bidx in kr_clean: 5267
Rows in df_model: 5415

Sample multi-child household in kr:
    v001  v002  bidx
1      1    24     1
2      1    24     2
10     2   100     1
11     2   100     2
15     2   199     1
16     2   199     1
28     5    17     1
29     5    17     1
35     6   100     2
36     6   100     1

df_model columns: ['v001', 'v002', 'hw70', 'stunted', 'v012_x', 'v106_x', 'v190_x', 'v025_x', 'hv009', 'bidx']
df_model head:
   v001  v002
0     1     9
1     1    24
2     1    24
3     1    39
4     1    69
5     1    84
6     1    99
7     1   160
8     2    43
9     2    86


In [9]:
# check if v001+v002+bidx+hw70 is unique in both files
print("kr_clean duplicates on v001+v002+bidx+hw70:",
      df_kr.duplicated(subset=["v001","v002","bidx","hw70"]).sum())

print("df_model duplicates on v001+v002+bidx+hw70:",
      df_model.duplicated(subset=["v001","v002","bidx","hw70"]).sum())

# confirm hw70 is in both
print("\nhw70 in df_kr:", "hw70" in df_kr.columns)
print("hw70 in df_model:", "hw70" in df_model.columns)

# look at the twin cases to understand the duplicate pattern
dup_mask = df_kr.duplicated(subset=["v001","v002","bidx"], keep=False)
print("\nDuplicate bidx cases:")
print(df_kr[dup_mask][["v001","v002","bidx","hw70","b4","b8"]].head(12))

kr_clean duplicates on v001+v002+bidx+hw70: 1
df_model duplicates on v001+v002+bidx+hw70: 1

hw70 in df_kr: True
hw70 in df_model: True

Duplicate bidx cases:
     v001  v002  bidx  hw70      b4   b8
15      2   199     1 -1.50  female  3.0
16      2   199     1  0.66  female  0.0
28      5    17     1 -0.31    male  4.0
29      5    17     1 -1.08  female  1.0
43      7    46     1 -0.34  female  3.0
44      7    46     1 -2.11  female  4.0
221    33   137     1  0.18    male  0.0
222    33   137     1 -2.98  female  2.0
231    35    58     1 -1.44  female  3.0
232    35    58     1  0.32  female  0.0
275    41    20     1 -0.47  female  1.0
276    41    20     1 -0.43  female  0.0


In [10]:
# remove the bidx column..... incorrectly added positionally in the last attempt
if "bidx" not in df_model.columns or df_model["bidx"].equals(df_kr["bidx"].iloc[:len(df_model)]):
    pass
df_model_clean = df_model.copy()

# merge on the four-part key: cluster + household + child index + haz score
# hw70 breaks ties between twins who share bidx
df_expanded = df_model_clean.merge(
    df_new,
    on=["v001", "v002", "bidx", "hw70"],
    how="left"
)

# one twin pair has identical haz scores, drop the resulting duplicate row
df_expanded = df_expanded.drop_duplicates(
    subset=["v001", "v002", "bidx", "hw70"]
)

print("Shape after merge and dedup:", df_expanded.shape)

rename_map = {
    "v001": "id_cluster",
    "v002": "id_household",
    "hw70": "haz_score",
    "stunted": "outcome_stunted",
    "v012_x": "und_maternal_age",
    "v106_x": "und_maternal_edu_level",
    "v190_x": "str_wealth_index",
    "v025_x": "str_residence",
    "hv009": "und_household_size",
    "sv005": "wt_sample_weight",
    "hw1": "imm_child_age_months",
    "b4": "imm_child_sex",
    "bord": "imm_birth_order",
    "b11": "imm_birth_interval",
    "m18": "imm_size_at_birth",
    "h11": "imm_had_diarrhea",
    "v133": "und_maternal_edu_years",
    "v437": "und_maternal_weight_kg",
    "v438": "und_maternal_height_cm",
    "v201": "und_total_children",
    "v024": "str_region",
    "v130": "str_religion",
    "bidx": "id_child_index"
}

df_expanded = df_expanded.rename(columns=rename_map)
cols_to_keep = list(rename_map.values()) + ["imm_first_born"]
df_expanded = df_expanded[cols_to_keep]

print("All columns:")
for col in df_expanded.columns:
    print(" ", col)

KeyError: 'hw70'

In [11]:
print("hw70 in df_model:", "hw70" in df_model.columns)
print("hw70 in df_new:", "hw70" in df_new.columns)
print("bidx in df_model:", "bidx" in df_model.columns)
print("bidx in df_new:", "bidx" in df_new.columns)
print("\ndf_new columns:", df_new.columns.tolist())
print("df_model columns:", df_model.columns.tolist())

hw70 in df_model: True
hw70 in df_new: False
bidx in df_model: True
bidx in df_new: True

df_new columns: ['v001', 'v002', 'bidx', 'sv005', 'hw1', 'b4', 'bord', 'b11', 'm18', 'h11', 'v133', 'v437', 'v438', 'v201', 'v024', 'v130', 'imm_first_born']
df_model columns: ['v001', 'v002', 'hw70', 'stunted', 'v012_x', 'v106_x', 'v190_x', 'v025_x', 'hv009', 'bidx']


In [12]:
df_model_clean = df_model.copy()

# bring hw70 into df_new so we can use it as a tiebreaker key
df_new_keyed = df_new.copy()
df_new_keyed["hw70"] = df_kr["hw70"].values

# merge on the four-part key
df_expanded = df_model_clean.merge(
    df_new_keyed,
    on=["v001", "v002", "bidx", "hw70"],
    how="left"
)

# drop the one twin pair with identical haz scores
df_expanded = df_expanded.drop_duplicates(
    subset=["v001", "v002", "bidx", "hw70"]
)

print("Shape after merge and dedup:", df_expanded.shape)

rename_map = {
    "v001": "id_cluster",
    "v002": "id_household",
    "hw70": "haz_score",
    "stunted": "outcome_stunted",
    "v012_x": "und_maternal_age",
    "v106_x": "und_maternal_edu_level",
    "v190_x": "str_wealth_index",
    "v025_x": "str_residence",
    "hv009": "und_household_size",
    "sv005": "wt_sample_weight",
    "hw1": "imm_child_age_months",
    "b4": "imm_child_sex",
    "bord": "imm_birth_order",
    "b11": "imm_birth_interval",
    "m18": "imm_size_at_birth",
    "h11": "imm_had_diarrhea",
    "v133": "und_maternal_edu_years",
    "v437": "und_maternal_weight_kg",
    "v438": "und_maternal_height_cm",
    "v201": "und_total_children",
    "v024": "str_region",
    "v130": "str_religion",
    "bidx": "id_child_index"
}

df_expanded = df_expanded.rename(columns=rename_map)
cols_to_keep = list(rename_map.values()) + ["imm_first_born"]
df_expanded = df_expanded[cols_to_keep]

print("All columns:")
for col in df_expanded.columns:
    print(" ", col)

Shape after merge and dedup: (5414, 24)
All columns:
  id_cluster
  id_household
  haz_score
  outcome_stunted
  und_maternal_age
  und_maternal_edu_level
  str_wealth_index
  str_residence
  und_household_size
  wt_sample_weight
  imm_child_age_months
  imm_child_sex
  imm_birth_order
  imm_birth_interval
  imm_size_at_birth
  imm_had_diarrhea
  und_maternal_edu_years
  und_maternal_weight_kg
  und_maternal_height_cm
  und_total_children
  str_region
  str_religion
  id_child_index
  imm_first_born


In [13]:
assert len(df_expanded) in [5414, 5415], f"Unexpected row count: {len(df_expanded)}"

print("=== Missingness ===")
print(df_expanded.isnull().mean().round(3))

print("\n=== Sampling weight check ===")
print("Missing weights:", df_expanded["wt_sample_weight"].isnull().sum())

print("\n=== Outcome distribution ===")
vc = df_expanded["outcome_stunted"].value_counts()
vcp = df_expanded["outcome_stunted"].value_counts(normalize=True).round(3)
print(pd.concat([vc, vcp], axis=1, keys=["count", "proportion"]))

print("\n=== Numeric summaries ===")
num_cols = [
    "imm_child_age_months", "imm_birth_interval", "imm_birth_order",
    "und_maternal_age", "und_maternal_edu_years",
    "und_maternal_weight_kg", "und_maternal_height_cm",
    "und_total_children", "und_household_size"
]
print(df_expanded[num_cols].describe().round(2))

# convert any remaining category dtypes before saving
for col in df_expanded.select_dtypes(["category"]).columns:
    df_expanded[col] = df_expanded[col].astype(str)

df_expanded.to_parquet(DATA_PROCESSED / "model_dataset_v2.parquet", index=False)
print("\nSaved: model_dataset_v2.parquet")
print("Final shape:", df_expanded.shape)

=== Missingness ===
id_cluster                0.0
id_household              0.0
haz_score                 0.0
outcome_stunted           0.0
und_maternal_age          0.0
und_maternal_edu_level    0.0
str_wealth_index          0.0
str_residence             0.0
und_household_size        0.0
wt_sample_weight          0.0
imm_child_age_months      0.0
imm_child_sex             0.0
imm_birth_order           0.0
imm_birth_interval        0.0
imm_size_at_birth         0.0
imm_had_diarrhea          0.0
und_maternal_edu_years    0.0
und_maternal_weight_kg    0.0
und_maternal_height_cm    0.0
und_total_children        0.0
str_region                0.0
str_religion              0.0
id_child_index            0.0
imm_first_born            0.0
dtype: float64

=== Sampling weight check ===
Missing weights: 0

=== Outcome distribution ===
                 count  proportion
outcome_stunted                   
0                 3482       0.643
1                 1932       0.357

=== Numeric summaries ==